# Chronicling America — Master Newspaper List (1869–1890)

Fetches every US newspaper title active during 1869–1890 from the [LOC Directory of US Newspapers](https://www.loc.gov/collections/directory-of-us-newspapers-in-american-libraries/) API.

In [2]:
import requests, time, re, os, json
import pandas as pd

BASE = "https://www.loc.gov/collections/directory-of-us-newspapers-in-american-libraries/"
PARAMS = {"fo": "json", "dates": "1869/1890", "c": 100}
MAX_RETRIES = 5
CHECKPOINT = "../data/processed/_loc_checkpoint.json"

# Resume from checkpoint if it exists
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        ckpt = json.load(f)
    results = ckpt["results"]
    page = ckpt["next_page"]
    print(f"Resuming from page {page} ({len(results):,} results already saved)")
else:
    results = []
    page = 1

while True:
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(BASE, params={**PARAMS, "sp": page}, timeout=60)
            if resp.status_code == 503:
                raise requests.exceptions.RequestException("503")
            resp.raise_for_status()
            data = resp.json()
            break
        except (requests.exceptions.RequestException, ValueError):
            if attempt < MAX_RETRIES - 1:
                wait = 3 * (attempt + 1)
                print(f"\n  Retry {attempt+1} for page {page} in {wait}s...")
                time.sleep(wait)
            else:
                print(f"\n\nFailed on page {page}. Checkpointed {len(results):,} results — re-run this cell to resume.")
                with open(CHECKPOINT, "w") as f:
                    json.dump({"results": results, "next_page": page}, f)
                raise SystemExit

    batch = data["results"]
    results.extend(batch)
    total = data["pagination"]["of"]
    print(f"\rPage {page}: {len(results):,} / {total:,}", end="", flush=True)

    # checkpoint every 10 pages
    if page % 10 == 0:
        with open(CHECKPOINT, "w") as f:
            json.dump({"results": results, "next_page": page + 1}, f)

    if len(results) >= total or not batch:
        break
    page += 1
    time.sleep(1)

# clean up checkpoint on success
if os.path.exists(CHECKPOINT):
    os.remove(CHECKPOINT)
print(f"\nDone. {len(results):,} titles fetched.")

Page 503: 50,287 / 50,287
Done. 50,287 titles fetched.


In [3]:
def parse_years(date_str):
    """Extract start/end years from date range like '[1869 TO 1890]'."""
    m = re.findall(r'\d{4}', date_str)
    if len(m) >= 2:
        return int(m[0]), int(m[1])
    elif len(m) == 1:
        return int(m[0]), None
    return None, None

rows = []
for r in results:
    dates_raw = r.get("dates", [""])[0] if r.get("dates") else ""
    yr_start, yr_end = parse_years(dates_raw)
    rows.append({
        "lccn":         r.get("number_lccn", [None])[0],
        "title":        r.get("title", ""),
        "state":        ", ".join(r.get("location_state", [])),
        "county":       ", ".join(r.get("location_county", [])),
        "city":         ", ".join(r.get("location_city", [])),
        "year_start":   yr_start,
        "year_end":     yr_end,
        "other_titles": ", ".join(r.get("other_title", [])),
    })

df = pd.DataFrame(rows)
print(f"{len(df):,} rows, {df['lccn'].nunique():,} unique LCCNs")
df.head(10)

50,287 rows, 50,287 unique LCCNs


,lccn,title,state,county,city,year_start,year_end,other_titles
0,sn94054224,"The 34th Ward Review (Kensington, Chicago, Ill...",illinois,cook,chicago,1889.0,1999.0,Thirty-fourth Ward review
1,sn91069488,"76 (Cincinnati, Ohio) 185?-18??",ohio,hamilton,cincinnati,1850.0,1899.0,Seventy-six'
2,sn94051479,A. Lusk & Co.'s Weekly San Francisco Trade Rev...,california,san francisco,san francisco,1800.0,1999.0,"San Francisco trade review, Trade review"
3,sn91059197,"Aamurusko (New York, Otter Tail Co., Minn.) 18...",minnesota,otter tail,new york mills,1884.0,1887.0,
4,sn91059198,"Aamurusko (New York Mills, Otter Tail Co., Min...","minnesota, oregon","otter tail, clatsop","new york mills, astoria",1887.0,1889.0,
5,2011229376,"The Abbeville Banner (Abbeville, S.C.) 18??-18...",south carolina,abbeville,abbeville,1800.0,1869.0,
6,2014218369,"The Abbeville Banner (Abbeville, C.H., S.C.) 1...",south carolina,abbeville,abbeville,1847.0,1869.0,
7,sn85026945,"The Abbeville Banner (Abbeville, S.C.) 1847-1869",south carolina,abbeville,abbeville,NaN,NaN,
8,sn84026854,"The Abbeville Medium (Abbeville, S.C.) 1871-1923",south carolina,abbeville,abbeville,1871.0,1923.0,
9,2014218604,"The Abbeville Messenger (Abbeville, S.C.) 1884...",south carolina,abbeville,abbeville,1884.0,1887.0,


In [4]:
df.to_csv("../data/processed/loc_newspaper_directory_1869_1890.csv", index=False)
print(f"Saved {len(df):,} rows to data/processed/loc_newspaper_directory_1869_1890.csv")

Saved 50,287 rows to data/processed/loc_newspaper_directory_1869_1890.csv
